# **FRAMEWORK V7: NOTEBOOK DE EXTRACCIÓN DE DATOS CRUDOS**

CAPA - INFORMACIÓN CLIMATICA

## **M0. Configuración General**



In [1]:
import pandas as pd
import requests
import time
from datetime import datetime

ANIO_INICIAL = 2015
ANIO_FINAL = 2025

VARIABLES = [

    "PRECTOTCORR",
    "T2M_MAX",
    "T2M_MIN",
    "T2M",
    "RH2M",
    "WS2M",
    "ALLSKY_SFC_SW_DWN"

]

#============================================================
# VARIABLES GLOBALES
#============================================================

dataset_maestro=[]

auditoria=[]

print("="*70)
print("FRAMEWORK V7")
print("INFORMACIÓN CLIMÁTICA")
print("INGESTA DE DATOS")
print("="*70)

print()

print("Fuente : NASA POWER")

print("Periodo:",ANIO_INICIAL,"-",ANIO_FINAL)

print()

FRAMEWORK V7
INFORMACIÓN CLIMÁTICA
INGESTA DE DATOS

Fuente : NASA POWER
Periodo: 2015 - 2025



## **M1. Definición de la Fuente Oficial**

In [2]:
nodos_cuenca = {

    "Villapinzon_Alta":{

        "lat":5.21,
        "lon":-73.60

    },

    "Tocancipa_Media":{

        "lat":4.96,
        "lon":-73.00

    },

    "Sopo_Sur":{

        "lat":4.70,
        "lon":-73.95

    },

    "Bogota_Sabana":{

        "lat":4.60,
        "lon":-74.15

    }

}

## **M2. Extracción y Procesamiento de Datos**

In [3]:
for nodo, coord in nodos_cuenca.items():
    print("-"*60)
    print("Nodo:", nodo)

    # Extracción de Datos
    url = (
        "https://power.larc.nasa.gov/api/temporal/monthly/point?"
        f"parameters={','.join(VARIABLES)}"
        "&community=AG"
        f"&longitude={coord['lon']}"
        f"&latitude={coord['lat']}"
        "&format=JSON"
        f"&start={ANIO_INICIAL}"
        f"&end={ANIO_FINAL}"
    )

    try:
        respuesta = requests.get(url, timeout=40)

        if not respuesta.ok:
            print("Error:", respuesta.status_code)
            continue

        datos = respuesta.json()
        parametros = datos["properties"]["parameter"]

        # Construcción del Dataset Crudo
        df = pd.DataFrame({
            "Fecha": pd.Series(parametros["PRECTOTCORR"]).index,
            "Precipitacion_mm": pd.Series(parametros["PRECTOTCORR"]).values,
            "Temp_Max_C": pd.Series(parametros["T2M_MAX"]).values,
            "Temp_Min_C": pd.Series(parametros["T2M_MIN"]).values,
            "Temp_Media_C": pd.Series(parametros["T2M"]).values,
            "Humedad_Relativa": pd.Series(parametros["RH2M"]).values,
            "Velocidad_Viento": pd.Series(parametros["WS2M"]).values,
            "Radiacion_Solar": pd.Series(parametros["ALLSKY_SFC_SW_DWN"]).values
        })

        # ELIMINAR RESUMEN ANUAL (YYYY13)
        df["Fecha"] = df["Fecha"].astype(str)
        df = df[df["Fecha"].str[-2:] != "13"].copy()

        # CONVERTIR FECHA
        df["Fecha"] = pd.to_datetime(df["Fecha"], format="%Y%m")

        # AGREGAR UBICACIÓN
        df["Nodo"] = nodo
        df["Latitud"] = coord["lat"]
        df["Longitud"] = coord["lon"]

        # Validación Inicial
        auditoria.append({
            "Nodo": nodo,
            "Registros": len(df),
            "Fecha_Inicial": df["Fecha"].min(),
            "Fecha_Final": df["Fecha"].max(),
            "Valores_Nulos": int(df.isnull().sum().sum()),
            "Duplicados": int(df.duplicated().sum()),
            "Precipitacion_Min": round(df["Precipitacion_mm"].min(), 2),
            "Precipitacion_Max": round(df["Precipitacion_mm"].max(), 2),
            "Temperatura_Min": round(df["Temp_Min_C"].min(), 2),
            "Temperatura_Max": round(df["Temp_Max_C"].max(), 2)
        })

        # Exportación (Nodos Individuales)
        nombre = f"Variables_Climaticas_{nodo}.xlsx"
        df.to_excel(nombre, index=False)
        dataset_maestro.append(df)

        print("Registros:", len(df))
        print("Archivo:", nombre)
        time.sleep(2) # Pausa para evitar saturar la API y por visibilidad

    except Exception as e:
        print("Error en el nodo", nodo, ":", e)

------------------------------------------------------------
Nodo: Villapinzon_Alta
Registros: 132
Archivo: Variables_Climaticas_Villapinzon_Alta.xlsx
------------------------------------------------------------
Nodo: Tocancipa_Media
Registros: 132
Archivo: Variables_Climaticas_Tocancipa_Media.xlsx
------------------------------------------------------------
Nodo: Sopo_Sur
Registros: 132
Archivo: Variables_Climaticas_Sopo_Sur.xlsx
------------------------------------------------------------
Nodo: Bogota_Sabana
Registros: 132
Archivo: Variables_Climaticas_Bogota_Sabana.xlsx


In [4]:
dataset_maestro=pd.concat(
    dataset_maestro,
    ignore_index=True
)

dataset_maestro=dataset_maestro.sort_values(
    ["Nodo","Fecha"]
)

print("DataFrame maestro consolidado creado y ordenado.")

DataFrame maestro consolidado creado y ordenado.


In [5]:
print()
print("="*70)
print("VALIDACIÓN DE INDEPENDENCIA")
print("="*70)

for i,n1 in enumerate(nodos_cuenca.keys()):

    for n2 in list(nodos_cuenca.keys())[i+1:]:

        s1=dataset_maestro[
            dataset_maestro.Nodo==n1
        ]["Precipitacion_mm"].reset_index(drop=True)

        s2=dataset_maestro[
            dataset_maestro.Nodo==n2
        ]["Precipitacion_mm"].reset_index(drop=True)

        print(
            f"{n1} vs {n2} -->",
            s1.equals(s2)
        )

print()


VALIDACIÓN DE INDEPENDENCIA
Villapinzon_Alta vs Tocancipa_Media --> False
Villapinzon_Alta vs Sopo_Sur --> False
Villapinzon_Alta vs Bogota_Sabana --> False
Tocancipa_Media vs Sopo_Sur --> False
Tocancipa_Media vs Bogota_Sabana --> False
Sopo_Sur vs Bogota_Sabana --> False



## **M3. Reporte de Auditoría**

In [6]:
auditoria=pd.DataFrame(auditoria)

auditoria.to_excel(
    "Auditoria_Capa_Climatica.xlsx",
    index=False
)

print()
print("Auditoría generada.")


Auditoría generada.


## **M4. Exportación de Resultados**

In [7]:
dataset_maestro.to_excel(
    "01_Capa_Climatica_V1.xlsx",
    index=False
)

print()
print("Archivo generado:")
print("01_Capa_Climatica_V1.xlsx")
print()
print("Registros:",len(dataset_maestro))
print("Variables:",len(dataset_maestro.columns))

print()
print("Auditoría generada. (Auditoria_Capa_Climatica.xlsx)")

print()
print("="*70)
print("PROCESO FINALIZADO")
print("="*70)
print()
print("Archivos generados")
print("----------------------------")
print("01_Capa_Climatica_V1.xlsx")
print("Auditoria_Capa_Climatica.xlsx")
print()
for nodo in nodos_cuenca.keys():
    print(f"Variables_Climaticas_{nodo}.xlsx")


Archivo generado:
01_Capa_Climatica_V1.xlsx

Registros: 528
Variables: 11

Auditoría generada. (Auditoria_Capa_Climatica.xlsx)

PROCESO FINALIZADO

Archivos generados
----------------------------
01_Capa_Climatica_V1.xlsx
Auditoria_Capa_Climatica.xlsx

Variables_Climaticas_Villapinzon_Alta.xlsx
Variables_Climaticas_Tocancipa_Media.xlsx
Variables_Climaticas_Sopo_Sur.xlsx
Variables_Climaticas_Bogota_Sabana.xlsx


## **M5. Resumen Final**

In [8]:
print()

print("="*70)
print("RESUMEN DE LA CAPA")
print("="*70)

print("Cobertura temporal:")

print(
    dataset_maestro["Fecha"].min(),
    "->",
    dataset_maestro["Fecha"].max()
)

print()

print("Número de nodos:",
      dataset_maestro["Nodo"].nunique())

print()

print("Valores nulos:",
      dataset_maestro.isnull().sum().sum())

print("Duplicados:",
      dataset_maestro.duplicated().sum())

print()

print("Estado: APROBADO")


RESUMEN DE LA CAPA
Cobertura temporal:
2015-01-01 00:00:00 -> 2025-12-01 00:00:00

Número de nodos: 4

Valores nulos: 0
Duplicados: 0

Estado: APROBADO
